# forecast.ipynb

- 목적: 연도별 기상 특예보 CSV를 하나로 합쳐 권역 기준으로 바로 비교 가능한 전처리본을 만든다.
- 입력: `fore2023.csv` ~ `fore2026.csv`
- 출력: `weather_total.csv`
- 메모: 실시간 기상 API를 붙이기 전, 과거 기상 흐름을 지역 단위로 비교하기 위한 준비 노트북이다.


## 1. 연도별 원본 불러오기
각 연도 CSV를 같은 형식으로 읽어 이후 결합 준비를 한다.


In [ ]:
# 연도별 기상 특예보 원본 CSV를 불러온다.
import pandas as pd

df2023 = pd.read_csv("data/fore2023.csv", encoding="cp949")
df2024 = pd.read_csv("data/fore2024.csv", encoding="cp949")
df2025 = pd.read_csv("data/fore2025.csv", encoding="cp949")
df2026 = pd.read_csv("data/fore2026.csv", encoding="cp949")


## 2. 데이터 결합과 기본 정리
연도별 데이터를 하나로 합치고 컬럼 공백, 제외 지역, 날짜형 변환을 처리한다.


In [ ]:
# 연도별 데이터를 하나로 묶고, 앱에서 쓰지 않을 지점을 제거한다.
weather = pd.concat([df2023, df2024, df2025, df2026], ignore_index=True)
weather.columns = weather.columns.str.strip()
weather = weather[~weather["지점명"].str.contains("울릉", na=False)]
weather["일시"] = pd.to_datetime(weather["일시"], errors="coerce")


## 3. 권역 매핑 기준 정의
원본 지점명을 앱에서 쓰는 시도/시군구 체계로 연결하는 사전을 만든다.


In [ ]:
# 원본 지점명을 앱 권역으로 매핑하기 위한 기준표를 정의한다.
region_map = {

"대구":("대구","대구"),
"부산":("부산","부산"),
"울산":("울산","울산"),

"포항":("경북","포항시"),
"경주시":("경북","경주시"),
"구미":("경북","구미시"),
"안동":("경북","안동시"),
"영주":("경북","영주시"),
"영천":("경북","영천시"),
"상주":("경북","상주시"),
"문경":("경북","문경시"),
"울진":("경북","울진군"),
"의성":("경북","의성군"),
"청송군":("경북","청송군"),
"봉화":("경북","봉화군"),
"영덕":("경북","영덕군"),

"창원":("경남","창원시"),
"북창원":("경남","창원시"),
"진주":("경남","진주시"),
"통영":("경남","통영시"),
"김해시":("경남","김해시"),
"밀양":("경남","밀양시"),
"양산시":("경남","양산시"),
"거제":("경남","거제시"),
"합천":("경남","합천군"),
"거창":("경남","거창군"),
"산청":("경남","산청군"),
"함양군":("경남","함양군"),
"의령군":("경남","의령군"),
"남해":("경남","남해군")

}


## 4. 권역 컬럼 생성
지점명을 기준으로 `지역`, `시군구` 컬럼을 새로 만들어 이후 비교에 쓰게 한다.


In [ ]:
# 지점명을 보고 시도/시군구를 추정해 새 컬럼으로 붙인다.
def match_region(x):

    for key in region_map:
        if key in str(x):
            return region_map[key]

    return (None, None)

weather[["지역", "시군구"]] = weather["지점명"].apply(
    lambda x: pd.Series(match_region(x))
)


## 5. 중복 제거와 최종 저장
불필요한 중복을 제거하고 최종 CSV로 내보낸다.


In [ ]:
# 중복 행을 제거해 저장 전 최종 데이터를 정리한다.
weather = weather.drop_duplicates()


In [ ]:
weather.to_csv(
    "data/weather_total.csv",
    index=False,
    encoding="utf-8-sig"
)

print("완료")
print("데이터 수:", len(weather))
